# Feature Extraction Baseline

# 1. Introdução

O objetivo principal é construir datasets tabulares baseline a partir dos arquivos temporais .txt da base de dados. Cada arquivo temporal é transformado em um vetor biomecânico representativo da marcha do paciente.

O resultado final será um dataset consolidado contendo:

- 1 linha por paciente;
- features biomecânicas agregadas;
- identificação do estudo (Ga, Ju, Si);
- metadados do experimento;
- labels clínicas apenas para validação futura.

# 2. Preparando o ambiente

Começamos importando algumas bibliotecas para começarmos com as construção da nossa base tabular.

In [1]:
import os
import glob
import numpy as np
import pandas as pd

from pathlib import Path

from google.colab import drive

Depois, importamos a base de dados Gait in Parkinson’s Disease diretamente do Google Drive.

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
root = Path('/content/drive/MyDrive')

matches = list(
    root.rglob('*gait-in-parkinsons-disease-1.0.0*')
)

for m in matches:
    print(m)

/content/drive/MyDrive/gait-in-parkinsons-disease-1.0.0


In [4]:
dataset_path = matches[0]

print(dataset_path)

/content/drive/MyDrive/gait-in-parkinsons-disease-1.0.0


# 3. Estrutura dos arquivos

Cada arquivo possui:

| Coluna | Descrição            |
| ------ | -------------------- |
| 0      | Tempo                |
| 1-8    | Sensores pé esquerdo |
| 9-16   | Sensores pé direito  |
| 17     | Total esquerdo       |
| 18     | Total direito        |

In [5]:
columns = [
    'time',

    'L1', 'L2', 'L3', 'L4',
    'L5', 'L6', 'L7', 'L8',

    'R1', 'R2', 'R3', 'R4',
    'R5', 'R6', 'R7', 'R8',

    'total_L',
    'total_R'
]

# 4. Definição das features biomecânicas para a baseline

Grupo 1 — Intensidade
- mean_total_L
- mean_total_R

Grupo 2 — Variabilidade
- std_total_L
- std_total_R
- cv_total_L
- cv_total_R

Grupo 3 — Assimetria
- assimetria_mean
- assimetria_abs

Grupo 4 — Amplitude
- peak_force_L
- peak_force_R

# 5. Função para extração das features na base

In [6]:
def extract_features(df):

    total_L = df['total_L']
    total_R = df['total_R']

    mean_total_L = total_L.mean()
    mean_total_R = total_R.mean()

    std_total_L = total_L.std()
    std_total_R = total_R.std()

    cv_total_L = std_total_L / mean_total_L
    cv_total_R = std_total_R / mean_total_R

    assimetria_mean = mean_total_L - mean_total_R

    assimetria_abs = abs(assimetria_mean)

    peak_force_L = total_L.max()
    peak_force_R = total_R.max()

    return {
        'mean_total_L': mean_total_L,
        'mean_total_R': mean_total_R,

        'std_total_L': std_total_L,
        'std_total_R': std_total_R,

        'cv_total_L': cv_total_L,
        'cv_total_R': cv_total_R,

        'assimetria_mean': assimetria_mean,
        'assimetria_abs': assimetria_abs,

        'peak_force_L': peak_force_L,
        'peak_force_R': peak_force_R
    }

# 6. Identifiação de subconjuntos

In [7]:
ga_files = list(dataset_path.rglob('*Ga*.txt'))
ju_files = list(dataset_path.rglob('*Ju*.txt'))
si_files = list(dataset_path.rglob('*Si*.txt'))

print(f'Total de arquivos Ga: {len(ga_files)}')
print(f'Total de arquivos Ju: {len(ju_files)}')
print(f'Total de arquivos Si: {len(si_files)}')

Total de arquivos Ga: 113
Total de arquivos Ju: 129
Total de arquivos Si: 64


# 7. Função de processamento dos pacientes

In [10]:
def process_dataset(file_list, dataset_name):

    dataset = []

    for file_path in file_list:

        try:

            df = pd.read_csv(
                file_path,
                sep='\\s+',
                header=None,
                names=columns
            )

            features = extract_features(df)

            patient_id = Path(file_path).stem

            features['dataset'] = dataset_name
            features['patient_id'] = patient_id

            dataset.append(features)

        except Exception as e:
            print(f'Erro no arquivo {file_path}: {e}')

    df = pd.DataFrame(dataset)

    ordered_columns = [
        'dataset',
        'patient_id',

        'mean_total_L',
        'mean_total_R',

        'std_total_L',
        'std_total_R',

        'cv_total_L',
        'cv_total_R',

        'assimetria_mean',
        'assimetria_abs',

        'peak_force_L',
        'peak_force_R'
    ]

    df = df[ordered_columns]

    return df

Agora, separamos cada um dos 3 grupos e processamos os dados

## Ga

In [11]:
ga_features_df = process_dataset(
    ga_files,
    'Ga'
)

## Ju

In [12]:
ju_features_df = process_dataset(
    ju_files,
    'Ju'
)

## Si

In [13]:
si_features_df = process_dataset(
    si_files,
    'Si'
)

# 8. Verificação do Dataset

## Ga

In [14]:
ga_features_df.head()

,dataset,patient_id,mean_total_L,mean_total_R,std_total_L,std_total_R,cv_total_L,cv_total_R,assimetria_mean,assimetria_abs,peak_force_L,peak_force_R
0,Ga,GaPt24_10,473.238435,502.094442,411.342104,422.206326,0.869207,0.840890,-28.856007,28.856007,1188.55,1210.77
1,Ga,GaCo09_02,470.210635,423.825689,409.763042,374.653720,0.871446,0.883981,46.384946,46.384946,1151.26,983.62
2,Ga,GaPt14_10,540.243175,425.180815,461.761851,389.215544,0.854730,0.915412,115.062360,115.062360,1276.00,1092.96
3,Ga,GaPt08_02,391.887131,419.585864,340.767717,370.100560,0.869556,0.882062,-27.698733,27.698733,904.86,927.74
4,Ga,GaPt28_10,591.748944,593.337949,488.596962,482.026601,0.825683,0.812398,-1.589005,1.589005,1324.84,1342.77


In [15]:
ga_features_df.shape

(113, 12)

## Ju

In [16]:
ju_features_df.head()

,dataset,patient_id,mean_total_L,mean_total_R,std_total_L,std_total_R,cv_total_L,cv_total_R,assimetria_mean,assimetria_abs,peak_force_L,peak_force_R
0,Ju,JuCo06_01,491.034501,458.574755,408.370356,387.052169,0.831653,0.844033,32.459747,32.459747,1003.64,1011.23
1,Ju,JuPt28_04,463.802448,441.058882,402.657635,390.279068,0.868166,0.884868,22.743566,22.743566,1014.09,1015.41
2,Ju,JuPt28_05,461.044277,439.617162,404.446082,392.266889,0.877239,0.892292,21.427115,21.427115,1018.93,1013.32
3,Ju,JuCo19_01,397.458941,384.621782,351.571417,335.652774,0.884548,0.872683,12.837160,12.837160,944.13,906.73
4,Ju,JuPt03_05,447.612376,449.400087,376.883198,395.509597,0.841986,0.880083,-1.787712,1.787712,1045.11,1016.07


In [17]:
ju_features_df.shape

(129, 12)

## Si

In [18]:
si_features_df.head()

,dataset,patient_id,mean_total_L,mean_total_R,std_total_L,std_total_R,cv_total_L,cv_total_R,assimetria_mean,assimetria_abs,peak_force_L,peak_force_R
0,Si,SiPt14_01,362.200837,390.913587,322.401322,333.247996,0.890118,0.852485,-28.712750,28.712750,877.47,937.31
1,Si,SiPt29_01,490.640455,529.150076,409.345073,428.998582,0.834308,0.810731,-38.509621,38.509621,1041.92,1085.15
2,Si,SiCo22_01,517.254519,509.153693,451.561998,443.610668,0.872998,0.871271,8.100825,8.100825,1371.59,1198.89
3,Si,SiCo01_01,361.433106,350.288006,309.265893,304.833984,0.855666,0.870238,11.145099,11.145099,791.67,787.71
4,Si,SiCo11_01,552.543424,515.343710,480.919473,453.572331,0.870374,0.880136,37.199714,37.199714,1336.94,1264.67


In [19]:
si_features_df.shape

(64, 12)

# 9. Verificação de valores ausentes

## Ga

In [20]:
print(ga_features_df.isnull().sum())

dataset            0
patient_id         0
mean_total_L       0
mean_total_R       0
std_total_L        0
std_total_R        0
cv_total_L         0
cv_total_R         0
assimetria_mean    0
assimetria_abs     0
peak_force_L       0
peak_force_R       0
dtype: int64


## Ju

In [21]:
print(ju_features_df.isnull().sum())

dataset            0
patient_id         0
mean_total_L       0
mean_total_R       0
std_total_L        0
std_total_R        0
cv_total_L         0
cv_total_R         0
assimetria_mean    0
assimetria_abs     0
peak_force_L       0
peak_force_R       0
dtype: int64


## Si

In [22]:
print(si_features_df.isnull().sum())

dataset            0
patient_id         0
mean_total_L       0
mean_total_R       0
std_total_L        0
std_total_R        0
cv_total_L         0
cv_total_R         0
assimetria_mean    0
assimetria_abs     0
peak_force_L       0
peak_force_R       0
dtype: int64


# 10. Criando uma pasta com a saída dos resultados

In [23]:
OUTPUT_PATH = Path('/content/drive/MyDrive/results')

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# 11. Salvando cada um dos datasets baseline

## Ga

In [24]:
ga_output = OUTPUT_PATH / 'ga_baseline_features.csv'

ga_features_df.to_csv(
    ga_output,
    index=False
)

print(f'Dataset salvo em: {ga_output}')

Dataset salvo em: /content/drive/MyDrive/results/ga_baseline_features.csv


## Ju

In [25]:
ju_output = OUTPUT_PATH / 'ju_baseline_features.csv'

ju_features_df.to_csv(
    ju_output,
    index=False
)

print(f'Dataset salvo em: {ju_output}')

Dataset salvo em: /content/drive/MyDrive/results/ju_baseline_features.csv


## Si

In [26]:
si_output = OUTPUT_PATH / 'si_baseline_features.csv'

si_features_df.to_csv(
    si_output,
    index=False
)

print(f'Dataset salvo em: {si_output}')

Dataset salvo em: /content/drive/MyDrive/results/si_baseline_features.csv
